In [1]:
import xml.etree.ElementTree as ET
import pandas as pd
import os
import time


In [ ]:
import os
import time
import pandas as pd
import xml.etree.ElementTree as ET

# -------------------------
# Settings
# -------------------------

xml_file = "/home/ubuntu/git/fennica/inst/examples/field_picking/Fennica_bibliot_20260413.xml"
out_file = "authors_700.csv"

chunk_size = 100_000

NS = {
    "marc": "http://www.loc.gov/MARC21/slim"
}

# Remove old output file if it already exists
if os.path.exists(out_file):
    os.remove(out_file)

# -------------------------
# Counters
# -------------------------

record_count = 0
author_row_count = 0
chunk_count = 0
chunk = []

start_time = time.time()

# -------------------------
# Parse XML
# -------------------------

for event, elem in ET.iterparse(xml_file, events=("end",)):
    if elem.tag.endswith("record"):

        record_count += 1

        # 001
        field_001 = None
        for cf in elem.findall("marc:controlfield", NS):
            if cf.attrib.get("tag") == "001":
                field_001 = cf.text.strip() if cf.text else None
                break

        # 035$a
        ids_035a = []
        for df035 in elem.findall("marc:datafield[@tag='035']", NS):
            for sf in df035.findall("marc:subfield[@code='a']", NS):
                if sf.text:
                    ids_035a.append(sf.text.strip())

        field_035a = " | ".join(ids_035a) if ids_035a else None

        # 700 fields
        for df700 in elem.findall("marc:datafield[@tag='700']", NS):

            sf_700a = []
            sf_700b = []
            sf_700c = []
            sf_700d = []
            sf_700e = []
            sf_7000 = []

            for sf in df700.findall("marc:subfield", NS):
                code = sf.attrib.get("code")
                value = sf.text.strip() if sf.text else None

                if not value:
                    continue

                if code == "a":
                    sf_700a.append(value)
                elif code == "b":
                    sf_700b.append(value)
                elif code == "c":
                    sf_700c.append(value)
                elif code == "d":
                    sf_700d.append(value)
                elif code == "e":
                    sf_700e.append(value)
                elif code == "0":
                    sf_7000.append(value)

            chunk.append({
                "field_001": field_001,
                "field_035a": field_035a,
                "author_700a": " | ".join(sf_700a) if sf_700a else None,
                "author_700b": " | ".join(sf_700b) if sf_700b else None,
                "author_700c": " | ".join(sf_700c) if sf_700c else None,
                "author_700d": " | ".join(sf_700d) if sf_700d else None,
                "author_700e": " | ".join(sf_700e) if sf_700e else None,
                "author_7000": " | ".join(sf_7000) if sf_7000 else None
            })

            author_row_count += 1

        # write chunk
        if len(chunk) >= chunk_size:
            chunk_count += 1

            pd.DataFrame(chunk).to_csv(
                out_file,
                mode="a",
                header=not os.path.exists(out_file),
                index=False,
                encoding="utf-8"
            )

            elapsed = time.time() - start_time
            print(
                f"Chunk {chunk_count} written | "
                f"records processed: {record_count:,} | "
                f"700 rows written: {author_row_count:,} | "
                f"elapsed: {elapsed/60:.1f} min"
            )

            chunk = []

        elem.clear()

# -------------------------
# Write remaining rows
# -------------------------

if chunk:
    chunk_count += 1

    pd.DataFrame(chunk).to_csv(
        out_file,
        mode="a",
        header=not os.path.exists(out_file),
        index=False,
        encoding="utf-8"
    )

    print(f"Final chunk {chunk_count} written.")

elapsed = time.time() - start_time

print("Done.")
print(f"Total records processed: {record_count:,}")
print(f"Total 700 rows written: {author_row_count:,}")
print(f"Output file: {out_file}")
print(f"Total time: {elapsed/60:.1f} min")

Chunk 1 written | records processed: 118,254 | 700 rows written: 100,000 | elapsed: 0.8 min
Chunk 2 written | records processed: 288,292 | 700 rows written: 200,002 | elapsed: 1.9 min
